# ERDOS-STRAUS SIEVE — Google Colab GPU Sieve
### Lead R&D: DaShawn (African American Developer & Mathematician)

This notebook is part of the **Ghost Braid** and **Substrate Delta Sieve** infrastructure. It is designed to run the CuPy-accelerated Erdős-Straus sieve up to $10^{10}$ in parallel batches utilizing Google Colab GPU instances (T4, L4, or A100 VRAM).

**Performance Note**: Multiples of 24 are processed in parallel blocks of 50,000 candidates using CUDA matrix calculations to check $X$ and $Y$ domains, writing checkpoints directly to Google Drive.

In [ ]:
# 1. MOUNT GOOGLE DRIVE
from google.colab import drive
import os, sys
from pathlib import Path

try:
    drive.mount('/content/drive')
    DRIVE_DIR = Path("/content/drive/MyDrive/erdos-straus-solver")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"[GHOST BRAID] Mounted Google Drive. Target output directory: {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = Path(".")
    print(f"[WARNING] Could not mount Google Drive. Running in local Colab space: {e}")

In [ ]:
# 2. GPU & CUPY DETECTOR
import subprocess

try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    gpu_name = r.stdout.strip() if r.returncode == 0 else 'CPU'
except Exception:
    gpu_name = 'CPU'

print(f"Hardware Substrate GPU: {gpu_name}")

try:
    import cupy as cp
    print(f"CuPy is pre-installed. Version: {cp.__version__}")
    print(f"CUDA Device Count: {cp.cuda.runtime.getDeviceCount()}")
except ImportError:
    print("CuPy not found. Installing appropriate cupy-cuda package...")
    try:
        cuda_version_raw = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
        if 'release 12' in cuda_version_raw:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x'])
        elif 'release 11' in cuda_version_raw:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'cupy-cuda11x'])
        else:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'cupy'])
        import cupy as cp
        print(f"Successfully installed CuPy: {cp.__version__}")
    except Exception as e:
        print(f"Failed to install CuPy. Falling back to NumPy CPU emulator: {e}")
        import numpy as cp
        cp.asnumpy = lambda x: np.array(x) if not isinstance(x, np.ndarray) else x

In [ ]:
# 3. ERDOS-STRAUS VECTORIZED SOLVER
import numpy as np
import math, json, time
from datetime import datetime

def sieve_batch(n_vals):
    results = []
    if not n_vals:
        return results

    # Transfer candidates to GPU memory
    N = cp.array(n_vals, dtype=cp.int64)
    mod9 = N % 9
    
    # Max probe range for hot corridor candidates is 200 elements
    probe_limits = cp.full_like(N, 200, dtype=cp.int32)
    MAX_PROBE = 200
    
    X_min = (N // 4) + 1
    X_max_limit = cp.minimum(3 * N // 4, cp.clip((N.astype(cp.float64)**2 * 0.3).astype(cp.int64), 0, 10_000_000_000))
    
    # Parallel evaluation of the X search space
    X_offset = cp.arange(MAX_PROBE, dtype=cp.int64)
    X = X_min[:, None] + X_offset[None, :]
    
    valid_X_mask = (X < X_max_limit[:, None]) & (X_offset[None, :] < probe_limits[:, None])
    X4 = 4 * X - N[:, None]
    valid_X_mask &= (X4 > 0)
    
    # Safe modulo check
    safe_X4 = cp.where(valid_X_mask, X4, 1)
    NX = N[:, None] * X
    valid_X_mask &= ((NX % safe_X4) == 0)
    
    # Extract surviving indices for Y search
    N_indices, X_offsets = cp.nonzero(valid_X_mask)
    
    N_idx_cpu = cp.asnumpy(N_indices)
    X_off_cpu = cp.asnumpy(X_offsets)
    N_host = cp.asnumpy(N)
    mod9_host = cp.asnumpy(mod9)
    X_min_host = cp.asnumpy(X_min)
    
    # Group solutions
    batch_solutions = {n: [] for n in n_vals}
    
    for i in range(len(N_idx_cpu)):
        idx = N_idx_cpu[i]
        n = int(N_host[idx])
        x = int(X_min_host[idx] + X_off_cpu[i])
        
        if len(batch_solutions[n]) >= 10:
            continue
            
        x4 = 4 * x - n
        if x4 <= 0: continue
        
        y_lim = min(int(2 * n * x / x4) + 1, int(n * n * 0.3))
        y_start = max(x, (n * x) // x4 + 1)
        
        for y in range(y_start, y_lim):
            denom = x4 * y - n * x
            if denom <= 0: continue
            if (n * x * y) % denom == 0:
                z = (n * x * y) // denom
                if z >= y:
                    batch_solutions[n].append((x, y, z))
                    if len(batch_solutions[n]) >= 10:
                        break
                        
    ts = datetime.now().isoformat()
    for n in n_vals:
        sols = sorted(list(set(batch_solutions[n])), key=lambda t: (t[0], t[1], t[2]))
        num_sol = len(sols)
        m9 = n % 9
        if num_sol > 0:
            depth = "BREACH_MOD9" if m9 in (0, 3, 6) else "STABLE_MOD9"
            results.append({
                "n": n, "mod9": m9, "mod24": 0,
                "depth": depth, "triple": list(sols[0]),
                "num_solutions": num_sol, "timestamp": ts
            })
        else:
            results.append({
                "n": n, "mod9": m9, "mod24": 0,
                "depth": "ANOMALY", "triple": [],
                "num_solutions": 0, "timestamp": ts
            })
            
    return results

print("[GHOST BRAID] Batch solver compiled on GPU substrate.")

In [ ]:
# 4. RESUME FROM Google Drive CHECKPOINT
JSONL_OUTPUT = DRIVE_DIR / "KAGGLE_OUTPUT_RECORD.jsonl"
STATE_FILE = DRIVE_DIR / "erdos_output.json"

DEFAULT_START_N = 4_300_000_000
DEFAULT_CHUNK_SIZE = 1_000_000_000 # 1 Billion Range
TARGET_LIMIT = 10_000_000_000

if STATE_FILE.exists():
    try:
        state = json.loads(STATE_FILE.read_text())
        start_n = state.get("last_n", DEFAULT_START_N)
        stats = state.get("stats", {"stable": 0, "breach": 0, "neutral": 0, "total_checked": 0})
        print(f"[GHOST BRAID] Resuming from checkpoint: n={start_n:,}")
        print(f"Stats: Stable={stats.get('stable',0):,}, Breach={stats.get('breach',0):,}")
    except Exception as e:
        print(f"[WARNING] Error reading checkpoint: {e}. Starting fresh.")
        start_n = DEFAULT_START_N
        stats = {"stable": 0, "breach": 0, "neutral": 0, "total_checked": 0, "total_solutions": 0}
else:
    print(f"[GHOST BRAID] No checkpoint found. Starting fresh from n={DEFAULT_START_N:,}")
    start_n = DEFAULT_START_N
    stats = {"stable": 0, "breach": 0, "neutral": 0, "total_checked": 0, "total_solutions": 0}

if "total_solutions" not in stats:
    stats["total_solutions"] = stats.get("stable", 0) + stats.get("breach", 0)

chunk_size = DEFAULT_CHUNK_SIZE
end_n = min(start_n + chunk_size, TARGET_LIMIT)
print(f"Target range: {start_n:,} -> {end_n:,}")

In [ ]:
# 5. MAIN GPU BATCH EXECUTION LOOP
BATCH_SIZE = 50000
SAVE_INTERVAL = 500_000  # Save to file every 500K candidates

n0 = ((start_n + 23) // 24) * 24
if n0 < start_n:
    n0 += 24

print(f"[RUNNING] Striding multiples of 24 starting from n={n0:,} up to {end_n:,}")
print(f"Batch size: {BATCH_SIZE:,} | Save interval: {SAVE_INTERVAL:,}")

run_start = time.time()
checkpoint_time = time.time()
candidates_at_last_save = stats["total_checked"]
solutions_buffer = []
anomalies = []

batch_n = []
try:
    for n in range(n0, end_n, 24):
        batch_n.append(n)
        
        if len(batch_n) >= BATCH_SIZE or n >= end_n - 24:
            # Execute GPU batch sifting
            results = sieve_batch(batch_n)
            
            # Write batch results to jsonl file and update stats
            with open(JSONL_OUTPUT, "a", encoding="utf-8") as f_jsonl:
                for r in results:
                    stats["total_checked"] += 1
                    
                    if r["depth"] != "ANOMALY":
                        stats["total_solutions"] += 1
                        if "STABLE" in r["depth"]:
                            stats["stable"] += 1
                        else:
                            stats["breach"] += 1
                            
                        # Buffer for recent solution tracking
                        solutions_buffer.append(r)
                        if len(solutions_buffer) > 1000:
                            solutions_buffer.pop(0)
                            
                        # Append directly to JSONL record
                        f_jsonl.write(json.dumps(r) + "\n")
                    else:
                        stats["neutral"] += 1
                        anomalies.append(r["n"])
            
            batch_n = []
            
            # Periodic Checkpoint Save
            candidates_since_save = stats["total_checked"] - candidates_at_last_save
            if candidates_since_save >= SAVE_INTERVAL:
                elapsed = time.time() - checkpoint_time
                rate = candidates_since_save / elapsed if elapsed > 0 else 0
                
                # Save state
                state = {
                    "last_n": n,
                    "solutions": solutions_buffer,
                    "stats": stats,
                    "rate_per_sec": round(rate),
                    "timestamp": datetime.now().isoformat(),
                    "gpu": gpu_name,
                    "anomalies": len(anomalies)
                }
                STATE_FILE.write_text(json.dumps(state, indent=2))
                
                progress_pct = (n - n0) / (end_n - n0) * 100
                print(f"  [{progress_pct:.1f}%] n={n:,} | Stats: S={stats['stable']} B={stats['breach']} | Rate={rate:.0f} cand/s")
                
                checkpoint_time = time.time()
                candidates_at_last_save = stats["total_checked"]

except KeyboardInterrupt:
    print("\n[INFO] Sieve execution interrupted by user.")
except Exception as e:
    print(f"\n[ERROR] Sieve execution failed: {e}")
    import traceback
    traceback.print_exc()

# Final Save
final_state = {
    "last_n": end_n,
    "solutions": solutions_buffer,
    "stats": stats,
    "anomalies": anomalies,
    "completed_chunk": True,
    "timestamp": datetime.now().isoformat(),
    "gpu": gpu_name,
    "chunk_size": chunk_size,
    "candidates_checked": stats["total_checked"]
}
STATE_FILE.write_text(json.dumps(final_state, indent=2))

total_runtime = time.time() - run_start
print("\n" + "=" * 60)
print("ERDOS-STRAUS SIEVE — RUN COMPLETE")
print(f"Total Runtime: {total_runtime/3600:.2f}h")
print(f"Candidates checked: {stats['total_checked']:,}")
print(f"Solutions found: {stats['total_solutions']:,}")
print(f"STABLE: {stats['stable']} | BREACH: {stats['breach']}")
print(f"ANOMALIES: {len(anomalies)}")
print("=" * 60)